# Notebook 31 — Experiment Tracking, Model Governance, and Release Engineering

    ## Learning objectives

    - Define immutable lineage across code, data, models, prompts, evaluation, and serving
- Build reproducibility manifests, model cards, promotion gates, and rollback records
- Separate technical evidence from ownership, approval, risk, privacy, and license decisions

    Cells labeled **optional GPU/remote** are deliberately guarded. Read them first,
    then opt in when the required hardware or Hugging Face Inference access is available.


In [ ]:
# Colab/local environment setup — run this cell first.
import importlib.util
import os
import platform
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
PACKAGES = []

if IN_COLAB and PACKAGES:
    print("Installing notebook dependencies in the Colab runtime...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *PACKAGES])

# Load an HF token from Colab Secrets without displaying it. In Colab, create a
# secret named HF_TOKEN (or HUGGINGFACE_TOKEN) and enable notebook access.
if IN_COLAB:
    from google.colab import userdata
    token = None
    for secret_name in ("HF_TOKEN", "HUGGINGFACE_TOKEN"):
        try:
            token = userdata.get(secret_name)
        except Exception:
            pass
        if token:
            break
    if token:
        os.environ["HF_TOKEN"] = token
        os.environ["HUGGINGFACE_TOKEN"] = token
else:
    try:
        from dotenv import load_dotenv
        load_dotenv(".env")
    except ImportError:
        pass

try:
    import torch
    accelerator = torch.cuda.get_device_name(0) if torch.cuda.is_available() else (
        "Apple MPS" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available() else "CPU"
    )
    print(f"runtime={platform.platform()} | Python={platform.python_version()} | accelerator={accelerator}")
    if False and not torch.cuda.is_available():
        print("WARNING: this training notebook is designed for a Colab GPU runtime. "
              "Select Runtime > Change runtime type > T4 GPU (or better).")
except ImportError:
    print(f"runtime={platform.platform()} | Python={platform.python_version()}")

print("Hugging Face token configured:", bool(os.getenv("HUGGINGFACE_TOKEN")))


## 31.1 The model is a dependency graph

A deployed LLM behavior is produced by weights, tokenizer, chat template, adapters, retrieval indexes,
prompts, tool schemas, decoding parameters, safety policies, inference engine, and hardware. Recording only
a model name cannot reproduce or govern the system. Every artifact should have an immutable identity and a
link to parents, tests, owners, and intended environments.

Experiment tracking answers what happened during a run. Governance answers whether an artifact may be built,
accessed, promoted, deployed, monitored, or retired—and who accepts residual risk. Tools can preserve evidence,
but cannot replace accountable review. Use the smallest process proportional to impact while maintaining the
same core lineage invariants.


In [ ]:
import hashlib, json, platform, sys
def sha256_bytes(value): return hashlib.sha256(value).hexdigest()
manifest = {
    "schema_version": 1, "code_commit": "replace-with-git-sha",
    "python": sys.version.split()[0], "platform": platform.platform(),
    "model": {"repo": "org/model", "revision": "immutable-commit"},
    "tokenizer": {"repo": "org/model", "revision": "immutable-commit"},
    "data": [{"dataset": "org/data", "revision": "immutable-commit", "split": "train"}],
    "config_sha256": sha256_bytes(b'{"learning_rate":5e-6}'),
}
print(json.dumps(manifest, indent=2))


## 31.2 Run records and reproducibility

Capture code commit and dirty-state diff, dependency lock, container digest, hardware/driver/runtime, seeds,
precision, distributed topology, environment variables by name but never secret values, input artifact hashes,
rendered configuration, commands, logs, metrics, checkpoints, and evaluation outputs. Store effective tokens
per update and tokens seen rather than relying on epochs. Test checkpoint resume and artifact reload.

Exact bitwise reproduction may be impossible across kernels or devices. State the required level: artifact
identity, metric tolerance, statistical conclusion, or bit equality. Repeat important experiments across seeds
and report uncertainty. A run that cannot be recreated can still provide evidence if its limitations are explicit;
silently claiming reproducibility is worse.


In [ ]:
required = {"schema_version", "code_commit", "python", "platform", "model", "tokenizer", "data", "config_sha256"}
missing = required - manifest.keys()
assert not missing, missing
canonical = json.dumps(manifest, sort_keys=True, separators=(",", ":")).encode()
print("manifest identity:", sha256_bytes(canonical))


## 31.3 Data, licensing, and privacy lineage

Dataset lineage includes sources, acquisition dates, licenses, consent or lawful basis where applicable,
transformations, filters, deduplication, splits, synthetic generators, human annotation guidance, removals,
and known contamination. Deletion must propagate to derived datasets, indexes, caches, and future rebuilds;
whether trained weights require remediation is a policy and legal question that needs documented escalation.

Model licenses can differ from data and code licenses and may impose use or redistribution conditions. Record
each dependency's terms and review compatibility before publication. Scan artifacts for credentials, personal
data, private prompts, and unexpected training examples. Access controls and retention should apply to run
traces as well as final weights because logs may contain raw data and model outputs.


In [ ]:
lineage = {
    "raw-v1": {"parents": [], "license": "record-me"},
    "filtered-v3": {"parents": ["raw-v1"], "transform": "filter@abc123"},
    "sft-v2": {"parents": ["filtered-v3"], "transform": "format@def456"},
    "adapter-v7": {"parents": ["sft-v2", "base@789"], "run": "run-0042"},
}
def ancestors(node):
    return set().union(*(ancestors(parent) | {parent} for parent in lineage.get(node, {}).get("parents", [])))
print("adapter ancestry:", ancestors("adapter-v7"))


## 31.4 Evaluation evidence and promotion gates

A promotion compares a candidate with the currently approved baseline on frozen target capability, general
retention, safety/security, calibration, subgroup slices, latency, throughput, and cost. Define thresholds and
statistical treatment before seeing results. Store raw predictions and grader versions. Deterministic validators,
human review, and calibrated model judges contribute different evidence.

Gates have owners and dispositions: pass, fail, approved exception with expiry, or insufficient evidence. Do not
average away a catastrophic safety failure with improvements elsewhere. Shadow tests and canaries evaluate the
assembled system in realistic traffic. A release record pins every component and links approvals, known risks,
monitoring queries, rollback target, and incident contacts.


In [ ]:
gates = [
    {"name":"task_accuracy", "value":.84, "minimum":.82, "blocking":True},
    {"name":"unsafe_rate", "value":.006, "maximum":.005, "blocking":True},
    {"name":"p95_seconds", "value":1.8, "maximum":2.0, "blocking":True},
]
def passes(g):
    return ("minimum" not in g or g["value"] >= g["minimum"]) and ("maximum" not in g or g["value"] <= g["maximum"])
print([(g["name"], passes(g)) for g in gates])
print("promote:", all(passes(g) or not g["blocking"] for g in gates))


## 31.5 Model cards, system cards, and accountability

A model card documents architecture, provenance, training, evaluations, intended uses, excluded uses, biases,
limitations, environmental/compute information, license, and contact. A system card expands to retrieval, tools,
safeguards, deployment context, threat model, human oversight, and observed incidents. Neither is marketing copy;
both should make negative evidence and uncertainty discoverable.

Assign owners for data, training, evaluation, security, privacy, legal review, infrastructure, and product risk.
Maintain change history and expiry dates. Independent reviewers need enough artifacts to reproduce key claims.
Risk classification and required approvals should depend on users, domain, autonomy, data sensitivity, and
consequence—not parameter count alone.


## 31.6 Rollout, monitoring, rollback, and retirement

Use immutable release bundles, staging, shadow traffic, small canaries, progressive exposure, and automatic
rollback criteria. Monitor quality proxies cautiously alongside errors, latency, token usage, refusals, retrieval
health, tool outcomes, drift, abuse, and security events. Protect telemetry with minimization, access, retention,
and redaction policies. Feedback data needs consent/provenance and must not flow directly into training.

Practice rollback while the system is healthy. Retain compatible previous weights, indexes, schemas, and engine
images. Retirement removes endpoints and credentials, updates inventories, applies retention/deletion policy,
and communicates downstream impact. Post-incident reviews should update tests, threat models, documentation,
and promotion gates so learning becomes durable institutional evidence.


## Exercises

    1. Create a manifest for one course model artifact and validate all immutable revisions.
2. Design blocking promotion gates and an expiring exception workflow.
3. Trace deletion of one source record through datasets, indexes, caches, and future training.
4. Write a rollback drill that includes tokenizer, prompt, retrieval index, and inference engine.

    ## Checkpoint

    Explain the notebook's central mechanism without using library names, then identify
    one assumption you would test before applying it to a real workload.
